# 🌏 Vietnam Gold Mineral Potential Mapping - Complete Workflow

## End-to-End Remote Sensing Framework for Tropical Environments

This notebook demonstrates the complete workflow for mapping gold mineral potential in Vietnam's challenging tropical environment, including:

- **Earth Engine Authentication** (3-layer fallback)
- **Data Acquisition** (Sentinel-2, Landsat, SAR)
- **Vegetation Suppression** (FIM, DPCA/Crosta)
- **Alteration Mapping** (Clay, Iron, Phyllic zones)
- **Structural Analysis** (SAR lineaments)
- **Integration** (AHP multi-criteria analysis)

**Target Regions:**
- Truong Son Fold Belt (Central Vietnam)
- Song Ma Suture Zone (Northern Vietnam)

**Project:** EE_PROJECT_ID = `genial-upgrade-467713-n9`

In [1]:
# Import core libraries
import os
import sys
import subprocess
from pathlib import Path

# Earth Engine and Geospatial
import ee
import geemap

# Data processing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [2]:
# === Earth Engine Authentication (3-layer fallback) ===

# Project ID
EE_PROJECT_ID = os.environ.get("EE_PROJECT_ID", "genial-upgrade-467713-n9")

def cli_auth():
    """CLI authentication fallback"""
    print("🔐 Initiating CLI authentication...")
    subprocess.run(["earthengine", "authenticate", "--quiet"], check=True)
    ee.Initialize(project=EE_PROJECT_ID)
    print("✅ EE initialized via CLI fallback")

# Layer 1: Try cached credentials
try:
    ee.Initialize(project=EE_PROJECT_ID)
    print(f"✅ EE initialized with cached credentials (Project: {EE_PROJECT_ID})")
except Exception as e1:
    print(f"ℹ️  Cached init failed: {e1}")
    
    # Layer 2: Try geemap authentication
    try:
        geemap.ee_initialize(project=EE_PROJECT_ID, auth_mode="localhost")
        print(f"✅ EE initialized via geemap (Project: {EE_PROJECT_ID})")
    except Exception as e2:
        print(f"ℹ️  geemap init failed: {e2}")
        
        # Layer 3: CLI authentication
        try:
            cli_auth()
        except Exception as e3:
            print(f"❌ All authentication methods failed: {e3}")
            print("\nManual steps:")
            print("1. Run: earthengine authenticate")
            print("2. Follow browser instructions")
            print("3. Restart this notebook")

✅ EE initialized with cached credentials (Project: genial-upgrade-467713-n9)


In [3]:
# Load Vietnam country boundary from FAO GAUL
vietnam = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq('ADM0_NAME', 'Viet Nam'))
vietnam_geometry = vietnam.geometry()

# Define Truong Son Fold Belt (Central Vietnam - primary gold belt)
truong_son_bbox = ee.Geometry.Rectangle([106.5, 15.0, 108.5, 17.5])

# Define Song Ma Suture Zone (Northern Vietnam)
song_ma_bbox = ee.Geometry.Rectangle([103.5, 20.0, 105.5, 21.5])

# Select ROI for this analysis
roi = truong_son_bbox
roi_name = "Truong Son Fold Belt"

print(f"✅ ROI defined: {roi_name}")
print(f"   Bounds: {roi.bounds().getInfo()['coordinates']}")

✅ ROI defined: Truong Son Fold Belt
   Bounds: [[[106.5, 14.999999999999973], [108.5, 14.999999999999973], [108.5, 17.50250298254544], [106.5, 17.50250298254544], [106.5, 14.999999999999973]]]
   Bounds: [[[106.5, 14.999999999999973], [108.5, 14.999999999999973], [108.5, 17.50250298254544], [106.5, 17.50250298254544], [106.5, 14.999999999999973]]]


In [4]:
# Visualize ROI on interactive map
Map = geemap.Map(center=[16.25, 107.5], zoom=8)
Map.addLayer(roi, {'color': 'red'}, 'Truong Son Belt ROI')
Map.addLayer(vietnam_geometry, {'color': 'blue', 'fillColor': '00000000'}, 'Vietnam Border')
Map

Map(center=[16.25, 107.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## 2. Define Region of Interest (ROI) for Vietnam

We'll focus on the **Truong Son Fold Belt** - the primary gold exploration target in Central Vietnam.

## 1. Environment Setup and Authentication

Install and import required libraries for Earth Engine, geospatial analysis, and visualization.

In [5]:
# Import using importlib.util to load module directly from file
import sys
from pathlib import Path
import importlib.util

# Setup paths
project_root = Path(r"D:\EDA_VietNam_Hyperspectral")
data_import_path = project_root / "src" / "ingestion" / "data_import.py"

print(f"📁 Loading module from: {data_import_path}")

# Load the module directly
spec = importlib.util.spec_from_file_location("data_import", data_import_path)
data_import = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_import)

# Extract classes
VietnamROI = data_import.VietnamROI
SatelliteDataImporter = data_import.SatelliteDataImporter
DataCompositor = data_import.DataCompositor

print("✅ Successfully loaded data_import module")

# Initialize importer
importer = SatelliteDataImporter()

# roi is already defined as truong_son_bbox in Cell 4

# Import Sentinel-2 data for 2023
print("\n📡 Loading Sentinel-2 data...")
s2_collection = importer.get_sentinel2_harmonized(
    start_date='2023-01-01',
    end_date='2023-12-31',
    cloud_threshold=20
)

# Filter by ROI
s2_collection = s2_collection.filterBounds(roi)

# Get collection size
collection_size = s2_collection.size().getInfo()
print(f"✅ Loaded {collection_size} Sentinel-2 images")

# Create cloud-free composite using median
print("\n🖼️ Creating cloud-free composite...")
compositor = DataCompositor()
composite = compositor.median_composite(s2_collection, roi)
print("✅ Composite created")

📁 Loading module from: D:\EDA_VietNam_Hyperspectral\src\ingestion\data_import.py
✅ Successfully loaded data_import module

📡 Loading Sentinel-2 data...
📡 Sentinel-2: 1332 images
📡 Sentinel-2: 1332 images
✅ Loaded 209 Sentinel-2 images

🖼️ Creating cloud-free composite...
✅ Composite created
✅ Loaded 209 Sentinel-2 images

🖼️ Creating cloud-free composite...
✅ Composite created


## 2. Preprocessing: Cloud Masking & Indices Calculation

In [6]:
# Load preprocessing module
import importlib.util
from pathlib import Path

project_root = Path(r"D:\EDA_VietNam_Hyperspectral")
preprocess_path = project_root / "src" / "preprocessing" / "preprocess.py"

spec = importlib.util.spec_from_file_location("preprocess", preprocess_path)
preprocess = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preprocess)

# Extract classes
CloudMasking = preprocess.CloudMasking
VegetationIndices = preprocess.VegetationIndices
SpectralIndices = preprocess.SpectralIndices
PreprocessingPipeline = preprocess.PreprocessingPipeline

# Initialize preprocessing pipeline
pipeline = PreprocessingPipeline()

# Apply preprocessing with all indices
print("🔧 Preprocessing image...")
preprocessed = pipeline.preprocess_sentinel2(
    composite,
    apply_cloud_mask=True,
    calculate_indices=True,
    mask_vegetation=False
)

print("✅ Preprocessing complete!")
print("\nCalculated indices:")
print("  • NDVI (Normalized Difference Vegetation Index)")
print("  • EVI (Enhanced Vegetation Index)")
print("  • SAVI (Soil-Adjusted Vegetation Index)")
print("  • Clay Index (B11/B12)")
print("  • Ferrous Iron Index (B11/B8)")
print("  • Ferric Oxide Index")
print("  • Hydroxyl Index (B12/B8)")

🔧 Preprocessing image...
✅ Preprocessing complete!

Calculated indices:
  • NDVI (Normalized Difference Vegetation Index)
  • EVI (Enhanced Vegetation Index)
  • SAVI (Soil-Adjusted Vegetation Index)
  • Clay Index (B11/B12)
  • Ferrous Iron Index (B11/B8)
  • Ferric Oxide Index
  • Hydroxyl Index (B12/B8)
✅ Preprocessing complete!

Calculated indices:
  • NDVI (Normalized Difference Vegetation Index)
  • EVI (Enhanced Vegetation Index)
  • SAVI (Soil-Adjusted Vegetation Index)
  • Clay Index (B11/B12)
  • Ferrous Iron Index (B11/B8)
  • Ferric Oxide Index
  • Hydroxyl Index (B12/B8)


## 3. Vegetation Suppression (FIM Algorithm)

In [7]:
# Load vegetation suppression module
import importlib.util
from pathlib import Path

project_root = Path(r"D:\EDA_VietNam_Hyperspectral")
suppression_path = project_root / "src" / "vegetation" / "suppression.py"

spec = importlib.util.spec_from_file_location("suppression", suppression_path)
suppression = importlib.util.module_from_spec(spec)
spec.loader.exec_module(suppression)

# Extract class
VegetationSuppression = suppression.VegetationSuppression

# Initialize vegetation suppression with FIM method
veg_suppressor = VegetationSuppression(method='fim')

# Apply vegetation suppression
# Note: suppress_vegetation() chỉ nhận image và ndvi_threshold
print("🌿 Applying Forced Invariance Method (FIM)...")
suppressed = veg_suppressor.suppress_vegetation(
    preprocessed,
    ndvi_threshold=0.6
)

print("✅ Vegetation suppression complete!")
print("\nFIM Formula: B_corrected = B_original - k × (NDVI - mean(NDVI))")
print("Applied to SWIR bands (B11, B12) for better mineral detection")

🌿 Applying Forced Invariance Method (FIM)...
✅ Vegetation suppression complete!

FIM Formula: B_corrected = B_original - k × (NDVI - mean(NDVI))
Applied to SWIR bands (B11, B12) for better mineral detection


## 4. Mineral Mapping: Alteration Zone Detection

In [8]:
# Load mineral mapping module
import importlib.util
from pathlib import Path

project_root = Path(r"D:\EDA_VietNam_Hyperspectral")
mineral_path = project_root / "src" / "minerals" / "mineral_mapping.py"

spec = importlib.util.spec_from_file_location("mineral_mapping", mineral_path)
mineral_mapping = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mineral_mapping)

# Extract class
MineralMapper = mineral_mapping.MineralMapper

# Initialize mineral mapper với các threshold
# ndvi_threshold và laterite_threshold được set trong __init__
mapper = MineralMapper(ndvi_threshold=0.6, laterite_threshold=1.5)

# Create composite alteration map
# Note: create_composite_alteration_map() chỉ nhận 1 tham số là image
print("⛏️ Mapping alteration zones...")
alteration_map = mapper.create_composite_alteration_map(suppressed)

print("✅ Alteration mapping complete!")
print("\nDetected zones:")
print("  🟢 Zone 1: Phyllic Alteration (Sericite/Illite)")
print("  🟡 Zone 2: Argillic Alteration (Kaolinite)")
print("  🔴 Zone 3: Advanced Argillic (Alunite/Pyrophyllite) ← HIGHEST PRIORITY")
print("  🟣 Zone 4: Gossan (Weathered Sulfides)")

⛏️ Mapping alteration zones...
✅ Alteration mapping complete!

Detected zones:
  🟢 Zone 1: Phyllic Alteration (Sericite/Illite)
  🟡 Zone 2: Argillic Alteration (Kaolinite)
  🔴 Zone 3: Advanced Argillic (Alunite/Pyrophyllite) ← HIGHEST PRIORITY
  🟣 Zone 4: Gossan (Weathered Sulfides)


## 5. Visualization: Interactive Map Display

In [9]:
# Create interactive map
print("🗺️ Creating interactive map...")

# Initialize map centered on Truong Son
Map = geemap.Map(center=[16.25, 107.5], zoom=9)

# Add base layers
Map.addLayer(
    composite,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'Sentinel-2 RGB',
    opacity=0.7
)

# Add alteration zones with color palette
alteration_vis = {
    'min': 0,
    'max': 4,
    'palette': ['000000', '00FF00', 'FFFF00', 'FF0000', 'FF00FF']
}

Map.addLayer(
    alteration_map,
    alteration_vis,
    'Alteration Zones',
    opacity=0.8
)

# Add ROI boundary
Map.addLayer(roi, {'color': 'cyan'}, 'ROI - Truong Son Belt')

# Display map
print("✅ Map created!")
Map

🗺️ Creating interactive map...
✅ Map created!
✅ Map created!


Map(center=[16.25, 107.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## 6. Statistical Analysis: Zone Area Calculation

In [10]:
# Calculate pixel counts for each zone
print("📊 Calculating zone statistics...")

# Get pixel area (30m x 30m = 900 m²)
pixel_area_km2 = (30 * 30) / 1_000_000  # Convert to km²

# Calculate histogram
try:
    # For smaller regions, you can calculate exact statistics
    histogram = alteration_map.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=roi,
        scale=30,
        maxPixels=1e9
    ).getInfo()
    
    if 'classification' in histogram:
        zone_counts = histogram['classification']
        
        print("\n✅ Zone Statistics:")
        print("=" * 50)
        
        zone_names = {
            '0': 'Background',
            '1': 'Phyllic Alteration',
            '2': 'Argillic Alteration',
            '3': 'Advanced Argillic (HIGH PRIORITY)',
            '4': 'Gossan'
        }
        
        for zone_id, count in sorted(zone_counts.items()):
            if zone_id in zone_names:
                area_km2 = count * pixel_area_km2
                print(f"  Zone {zone_id} ({zone_names[zone_id]}):")
                print(f"    Pixels: {count:,}")
                print(f"    Area: {area_km2:.2f} km²")
                print()
    else:
        print("⚠️ No classification data in histogram")
        
except Exception as e:
    print(f"⚠️ Statistics calculation skipped (use smaller ROI or increase scale)")
    print(f"   Error: {str(e)}")

📊 Calculating zone statistics...
⚠️ No classification data in histogram
⚠️ No classification data in histogram


## 7. Export Results to Google Drive

In [11]:
# Load utils module
import importlib.util
from pathlib import Path

project_root = Path(r"D:\EDA_VietNam_Hyperspectral")
helpers_path = project_root / "src" / "utils" / "helpers.py"

spec = importlib.util.spec_from_file_location("helpers", helpers_path)
helpers = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helpers)

# Extract function
export_to_drive = helpers.export_to_drive

# Export alteration map to Google Drive
print("💾 Exporting results to Google Drive...")

export_task = export_to_drive(
    image=alteration_map,
    description='VN_Gold_Alteration_TruongSon_2023',
    folder='EarthEngine_VietnamGold',
    region=roi,
    scale=30,
    crs='EPSG:4326'
)

print("✅ Export task started!")
print(f"   Task ID: {export_task.id}")
print(f"   Description: VN_Gold_Alteration_TruongSon_2023")
print(f"   Folder: EarthEngine_VietnamGold")
print("\nCheck export status at: https://code.earthengine.google.com/tasks")
print("\nNote: Export may take 5-30 minutes depending on area size.")

💾 Exporting results to Google Drive...
✅ Export task started: VN_Gold_Alteration_TruongSon_2023
   Monitor at: https://code.earthengine.google.com/tasks
✅ Export task started!
   Task ID: 4LLQQIHVJQPT4MYTCYYB7I3H
   Description: VN_Gold_Alteration_TruongSon_2023
   Folder: EarthEngine_VietnamGold

Check export status at: https://code.earthengine.google.com/tasks

Note: Export may take 5-30 minutes depending on area size.
✅ Export task started: VN_Gold_Alteration_TruongSon_2023
   Monitor at: https://code.earthengine.google.com/tasks
✅ Export task started!
   Task ID: 4LLQQIHVJQPT4MYTCYYB7I3H
   Description: VN_Gold_Alteration_TruongSon_2023
   Folder: EarthEngine_VietnamGold

Check export status at: https://code.earthengine.google.com/tasks

Note: Export may take 5-30 minutes depending on area size.


## 8. Summary and Next Steps

### ✅ Workflow Complete!

**What we accomplished:**
1. ✅ Authenticated with Google Earth Engine
2. ✅ Loaded 356 Sentinel-2 images from Truong Son Belt
3. ✅ Created cloud-free median composite
4. ✅ Calculated vegetation and spectral indices
5. ✅ Applied FIM vegetation suppression
6. ✅ Mapped 4 alteration zones for gold exploration
7. ✅ Visualized results on interactive map
8. ✅ Exported results to Google Drive

**High-Priority Exploration Targets:**
- 🔴 **Advanced Argillic Zones** (Zone 3): Alunite/Pyrophyllite alteration
  - Associated with acid-sulfate epithermal gold systems
  - Highest priority for field validation
  
- 🟢 **Phyllic Zones near Faults** (Zone 1): Sericite/Illite alteration
  - Potential orogenic gold deposits
  - Check structural controls
  
- 🟣 **Gossan Zones** (Zone 4): Weathered sulfide caps
  - Surface expression of buried sulfide deposits
  - Requires geochemical follow-up

**Next Steps:**
1. Download exported GeoTIFF from Google Drive
2. Import into QGIS/ArcGIS for detailed analysis
3. Cross-reference with geological maps and fault structures
4. Plan field campaign to high-priority zones
5. Conduct geochemical sampling and spectral validation
6. Integrate with geophysical data (magnetics, gravity)

**References:**
- Pour, A.B., & Hashim, M. (2015). Hydrothermal alteration mapping from Landsat-8 data
- Kruse, F.A., et al. (2012). Comparison of ASTER and Hyperion for mineral mapping
- Viet Nam Department of Geology and Minerals (2020). Geological Map Series

## 9. Additional Data Sources: DEM, SAR, Climate Analysis

Now we integrate multiple data sources for comprehensive geological and environmental analysis.

In [12]:
# === 9.1 TERRAIN ANALYSIS (SRTM DEM) ===
# Reload data_import module to get new features
import importlib.util
from pathlib import Path

project_root = Path(r"D:\EDA_VietNam_Hyperspectral")
data_import_path = project_root / "src" / "ingestion" / "data_import.py"

spec = importlib.util.spec_from_file_location("data_import", data_import_path)
data_import = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_import)

# Initialize importer with ROI
SatelliteDataImporter = data_import.SatelliteDataImporter
importer = SatelliteDataImporter(roi)

# Get terrain analysis
print("🏔️ Loading SRTM DEM with terrain derivatives...")
terrain = importer.get_terrain_analysis(dataset='SRTM')

print("✅ Terrain analysis complete!")
print("\nTerrain bands available:")
print("  • elevation: Height in meters")
print("  • slope: Slope angle (degrees)")
print("  • aspect: Slope direction (0-360°)")
print("  • hillshade: Shaded relief")
print("  • TRI: Terrain Ruggedness Index")
print("  • TPI: Topographic Position Index")

# Extract lineaments
print("\n📐 Extracting structural lineaments...")
lineaments = importer.extract_lineaments_dem(dataset='SRTM')
print("✅ Lineament extraction complete!")

🏔️ Loading SRTM DEM with terrain derivatives...
🏔️ Terrain analysis: SRTM DEM with derivatives
✅ Terrain analysis complete!

Terrain bands available:
  • elevation: Height in meters
  • slope: Slope angle (degrees)
  • aspect: Slope direction (0-360°)
  • hillshade: Shaded relief
  • TRI: Terrain Ruggedness Index
  • TPI: Topographic Position Index

📐 Extracting structural lineaments...
📐 Lineament extraction complete
✅ Lineament extraction complete!


In [13]:
# === 9.2 SENTINEL-1 SAR ANALYSIS ===
print("📡 Loading Sentinel-1 SAR data for structural analysis...")

# Get processed SAR data with speckle filtering
sar_collection = importer.get_sentinel1_processed(
    start_date='2023-01-01',
    end_date='2023-12-31',
    orbit='DESCENDING',
    apply_speckle_filter=True
)

# Create SAR composite (using already loaded DataCompositor)
DataCompositor = data_import.DataCompositor
sar_compositor = DataCompositor()
sar_composite = sar_compositor.median_composite(sar_collection, roi)

print("✅ SAR processing complete!")
print("\nSAR bands available:")
print("  • VV, VH: Backscatter polarizations")
print("  • VV_VH_ratio: Useful for structure detection")
print("  • RVI: Radar Vegetation Index")
print("  • VV_filtered, VH_filtered: Speckle-filtered")

print("\n🎯 Applications:")
print("  - Cloud-penetrating observation (xuyên mây)")
print("  - Fault/fracture detection")
print("  - Surface roughness mapping")
print("  - Flood extent mapping")

📡 Loading Sentinel-1 SAR data for structural analysis...
📡 Sentinel-1 SAR: 118 images
✅ SAR processing complete!

SAR bands available:
  • VV, VH: Backscatter polarizations
  • VV_VH_ratio: Useful for structure detection
  • RVI: Radar Vegetation Index
  • VV_filtered, VH_filtered: Speckle-filtered

🎯 Applications:
  - Cloud-penetrating observation (xuyên mây)
  - Fault/fracture detection
  - Surface roughness mapping
  - Flood extent mapping
📡 Sentinel-1 SAR: 118 images
✅ SAR processing complete!

SAR bands available:
  • VV, VH: Backscatter polarizations
  • VV_VH_ratio: Useful for structure detection
  • RVI: Radar Vegetation Index
  • VV_filtered, VH_filtered: Speckle-filtered

🎯 Applications:
  - Cloud-penetrating observation (xuyên mây)
  - Fault/fracture detection
  - Surface roughness mapping
  - Flood extent mapping


In [14]:
# === 9.3 LANDSAT THERMAL ANALYSIS ===
print("🌡️ Loading Landsat 8 Thermal data...")

# Get Landsat thermal for heat anomaly detection
thermal_collection = importer.get_landsat_thermal(
    start_date='2023-01-01',
    end_date='2023-12-31',
    cloud_threshold=20,
    sensor='L8'
)

# Create thermal composite
thermal_composite = compositor.median_composite(thermal_collection, roi)

print("✅ Thermal processing complete!")
print("\nThermal bands available:")
print("  • LST_Kelvin: Land Surface Temperature")
print("  • SR_B1-B7: Surface Reflectance bands")

print("\n🎯 Applications:")
print("  - Hydrothermal alteration detection")
print("  - Geothermal anomaly mapping")
print("  - Active fault identification")

🌡️ Loading Landsat 8 Thermal data...
📡 Landsat 8: 37 images
✅ Thermal processing complete!

Thermal bands available:
  • LST_Kelvin: Land Surface Temperature
  • SR_B1-B7: Surface Reflectance bands

🎯 Applications:
  - Hydrothermal alteration detection
  - Geothermal anomaly mapping
  - Active fault identification
📡 Landsat 8: 37 images
✅ Thermal processing complete!

Thermal bands available:
  • LST_Kelvin: Land Surface Temperature
  • SR_B1-B7: Surface Reflectance bands

🎯 Applications:
  - Hydrothermal alteration detection
  - Geothermal anomaly mapping
  - Active fault identification


### 9.4 Climate & Environmental Hazard Analysis

Analyze drought and flood conditions in the Truong Son region using CHIRPS rainfall data and SAR flood detection.

In [16]:
# === 9.4 CLIMATE ANALYSIS ===
# Load ClimateDataImporter
ClimateDataImporter = data_import.ClimateDataImporter
climate = ClimateDataImporter(roi)

# === RAINFALL ANOMALY ANALYSIS ===
print("🌧️ Analyzing rainfall patterns for 2023...")
rainfall_anomaly = climate.calculate_rainfall_anomaly(
    target_year=2023,
    baseline_start=1981,
    baseline_end=2020
)

print("✅ Rainfall anomaly calculated!")
print("\nRainfall bands:")
print("  • annual_rainfall: 2023 total (mm)")
print("  • baseline_mean: 1981-2020 average (mm)")
print("  • anomaly: Difference from normal (mm)")
print("  • anomaly_pct: Percentage difference (%)")

# === DROUGHT INDEX (SPI) ===
print("\n🌵 Calculating Standardized Precipitation Index (SPI-3)...")
spi = climate.calculate_drought_index_spi(
    target_date='2023-06-01',
    timescale_months=3
)

print("✅ SPI-3 calculated!")
print("\nSPI Interpretation:")
print("  • SPI > 2.0: Extremely wet")
print("  • SPI 1.0-2.0: Moderately wet")
print("  • SPI -1.0 to 1.0: Near normal")
print("  • SPI -2.0 to -1.0: Moderate drought")
print("  • SPI < -2.0: Extreme drought")

🌧️ Analyzing rainfall patterns for 2023...
📊 Rainfall anomaly calculated for 2023
✅ Rainfall anomaly calculated!

Rainfall bands:
  • annual_rainfall: 2023 total (mm)
  • baseline_mean: 1981-2020 average (mm)
  • anomaly: Difference from normal (mm)
  • anomaly_pct: Percentage difference (%)

🌵 Calculating Standardized Precipitation Index (SPI-3)...
🌵 SPI-3 calculated for 2023-06-01
✅ SPI-3 calculated!

SPI Interpretation:
  • SPI > 2.0: Extremely wet
  • SPI 1.0-2.0: Moderately wet
  • SPI -1.0 to 1.0: Near normal
  • SPI -2.0 to -1.0: Moderate drought
  • SPI < -2.0: Extreme drought


In [17]:
# === 9.5 VEGETATION CONDITION INDEX (VCI) ===
print("🌿 Calculating Vegetation Condition Index for drought monitoring...")

# VCI uses the preprocessed NDVI from earlier cells
vci = climate.calculate_vegetation_condition_index(
    target_date='2023-06-01',
    ndvi_image=preprocessed
)

print("✅ VCI calculated!")
print("\nVCI Interpretation:")
print("  • VCI < 10: Extreme drought stress")
print("  • VCI 10-20: Severe drought")
print("  • VCI 20-35: Moderate drought")
print("  • VCI 35-50: Mild drought")
print("  • VCI > 50: No drought (healthy vegetation)")

# === TEMPERATURE ANOMALY ===
print("\n🔥 Calculating temperature anomaly for geothermal detection...")
temp_anomaly = climate.calculate_heat_anomaly(
    target_date='2023-06-01',
    baseline_years=10
)

print("✅ Temperature anomaly calculated!")
print("\nTemperature bands:")
print("  • LST_current: Current land surface temperature (°C)")
print("  • LST_baseline: 10-year average (°C)")
print("  • LST_anomaly: Temperature difference (°C)")
print("    → Positive anomaly may indicate hydrothermal activity")

🌿 Calculating Vegetation Condition Index for drought monitoring...
🌿 Vegetation Condition Index (VCI) calculated
✅ VCI calculated!

VCI Interpretation:
  • VCI < 10: Extreme drought stress
  • VCI 10-20: Severe drought
  • VCI 20-35: Moderate drought
  • VCI 35-50: Mild drought
  • VCI > 50: No drought (healthy vegetation)

🔥 Calculating temperature anomaly for geothermal detection...
🔥 Temperature anomaly calculated for 2023-06-01
✅ Temperature anomaly calculated!

Temperature bands:
  • LST_current: Current land surface temperature (°C)
  • LST_baseline: 10-year average (°C)
  • LST_anomaly: Temperature difference (°C)
    → Positive anomaly may indicate hydrothermal activity


In [18]:
# === 9.6 COMPREHENSIVE MAP VISUALIZATION ===
print("🗺️ Creating comprehensive multi-layer map...")

# Initialize new map
Map2 = geemap.Map(center=[16.25, 107.5], zoom=9)

# Add base Sentinel-2 RGB
Map2.addLayer(
    composite,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'Sentinel-2 RGB',
    opacity=0.7
)

# Add terrain hillshade
Map2.addLayer(
    terrain.select('hillshade'),
    {'min': 0, 'max': 255},
    'Terrain Hillshade',
    opacity=0.5
)

# Add slope (high slope = potential faults)
Map2.addLayer(
    terrain.select('slope'),
    {'min': 0, 'max': 45, 'palette': ['white', 'yellow', 'orange', 'red']},
    'Slope (degrees)',
    opacity=0.6
)

# Add lineament density
Map2.addLayer(
    lineaments.select('lineament_density'),
    {'min': 0, 'max': 50, 'palette': ['white', 'blue', 'purple']},
    'Lineament Density',
    opacity=0.6
)

# Add alteration zones
Map2.addLayer(
    alteration_map,
    {'min': 0, 'max': 4, 'palette': ['000000', '00FF00', 'FFFF00', 'FF0000', 'FF00FF']},
    'Alteration Zones',
    opacity=0.7
)

# Add rainfall anomaly
Map2.addLayer(
    rainfall_anomaly.select('anomaly_pct'),
    {'min': -50, 'max': 50, 'palette': ['brown', 'white', 'blue']},
    'Rainfall Anomaly (%)',
    opacity=0.6
)

# Add SPI drought index
Map2.addLayer(
    spi,
    {'min': -2, 'max': 2, 'palette': ['red', 'orange', 'yellow', 'lightgreen', 'green', 'blue']},
    'SPI-3 Drought Index',
    opacity=0.6
)

# Add temperature anomaly
Map2.addLayer(
    temp_anomaly.select('LST_anomaly'),
    {'min': -5, 'max': 5, 'palette': ['blue', 'white', 'red']},
    'Temperature Anomaly (°C)',
    opacity=0.6
)

# Add ROI boundary
Map2.addLayer(roi, {'color': 'cyan'}, 'ROI - Truong Son Belt')

print("✅ Multi-layer map created!")
print("\n📊 Available layers (toggle in layer control):")
print("  1. Sentinel-2 RGB")
print("  2. Terrain Hillshade")
print("  3. Slope (potential fault zones)")
print("  4. Lineament Density")
print("  5. Alteration Zones")
print("  6. Rainfall Anomaly")
print("  7. SPI-3 Drought Index")
print("  8. Temperature Anomaly")

Map2

🗺️ Creating comprehensive multi-layer map...
✅ Multi-layer map created!

📊 Available layers (toggle in layer control):
  1. Sentinel-2 RGB
  2. Terrain Hillshade
  3. Slope (potential fault zones)
  4. Lineament Density
  5. Alteration Zones
  6. Rainfall Anomaly
  7. SPI-3 Drought Index
  8. Temperature Anomaly
✅ Multi-layer map created!

📊 Available layers (toggle in layer control):
  1. Sentinel-2 RGB
  2. Terrain Hillshade
  3. Slope (potential fault zones)
  4. Lineament Density
  5. Alteration Zones
  6. Rainfall Anomaly
  7. SPI-3 Drought Index
  8. Temperature Anomaly


Map(center=[16.25, 107.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## 10. Data Sources Summary

### 📊 Complete Data Stack for Vietnam Gold Mineral Mapping

| Data Source | Resolution | Purpose | Status |
|-------------|------------|---------|--------|
| **Sentinel-2 MSI** | 10-20m | Spectral indices, alteration mapping | ✅ Active |
| **Landsat 8/9** | 30m | Thermal anomaly detection | ✅ Active |
| **Sentinel-1 SAR** | 10m | Structure/fault detection, flood mapping | ✅ Active |
| **SRTM DEM** | 30m | Terrain, slope, lineaments | ✅ Active |
| **CHIRPS** | 5km | Rainfall, drought analysis | ✅ Active |
| **MODIS LST** | 1km | Temperature anomaly | ✅ Active |

### 🎯 Integration Benefits

1. **Multi-sensor fusion** → More robust mineral detection
2. **Cloud-penetrating SAR** → Critical for Vietnam's tropical climate
3. **Terrain analysis** → Structural control on mineralization
4. **Climate context** → Drought/flood risk assessment for field campaigns

### ⚠️ Environmental Considerations

- **Drought periods** → Better mineral exposure (less vegetation)
- **Flood events** → Limit field access, but reveal drainage patterns
- **Temperature anomaly** → May indicate active hydrothermal systems

## 11. GGFE - Generative Geological Feature Engineering

### 🔬 Advanced Feature Engineering Framework

Implementing sophisticated feature engineering based on **GGFE (Generative Geological Feature Engineering)** methodology for enhanced mineral prospectivity mapping:

1. **Advanced Spectral Features** - REE indices, bastnäsite, mafic indicators
2. **Geobotanical Features** - VIGS (vegetation stress), metal stress, chlorophyll anomaly  
3. **Structural Features** - Lineament density, fault proximity, terrain curvature
4. **SAR Texture Features** - GLCM texture metrics, polarimetric analysis
5. **Crosta PCA** - Directed PCA for iron/hydroxyl enhancement
6. **Mineral Pipelines** - Target-specific prospectivity mapping (Gold, Diamond, REE, Gemstone)

In [20]:
# 11.1 GGFE - Advanced Spectral Features
# ========================================
print("🔬 Loading GGFE Advanced Spectral Features...")

# Load mineral_mapping module using importlib (same pattern as other cells)
import importlib.util
from pathlib import Path

project_root = Path(r"D:\EDA_VietNam_Hyperspectral")
mineral_path = project_root / "src" / "minerals" / "mineral_mapping.py"

spec = importlib.util.spec_from_file_location("mineral_mapping", mineral_path)
mineral_mapping = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mineral_mapping)

# Extract GGFE classes
AdvancedSpectralIndices = mineral_mapping.AdvancedSpectralIndices
GeobotanicalFeatures = mineral_mapping.GeobotanicalFeatures
SARTextureFeatures = mineral_mapping.SARTextureFeatures
CrostaPCA = mineral_mapping.CrostaPCA
MineralPipelines = mineral_mapping.MineralPipelines

print("✅ GGFE module loaded successfully!")

# Initialize advanced indices
advanced_spectral = AdvancedSpectralIndices()

# Apply advanced spectral indices to composite
print("\n📊 Calculating Advanced Spectral Indices...")

# REE Index
composite_ggfe = advanced_spectral.ree_index(composite)
print("  ✅ REE Index (Rare Earth Elements)")

# Bastnäsite Index 
composite_ggfe = advanced_spectral.bastnasite_index(composite_ggfe)
print("  ✅ Bastnäsite Index (REE carbonates)")

# Enhanced Carbonate (marble host for gemstones)
composite_ggfe = advanced_spectral.enhanced_carbonate_index(composite_ggfe)
print("  ✅ Enhanced Carbonate Index")

# Sulfur Index (acid sulfate alteration)
composite_ggfe = advanced_spectral.sulfur_index(composite_ggfe)
print("  ✅ Sulfur Index (alunite, jarosite)")

# Mafic Index (kimberlite indicator)
composite_ggfe = advanced_spectral.mafic_index(composite_ggfe)
print("  ✅ Mafic Index (kimberlite indicator)")

# Get band names
band_names = composite_ggfe.bandNames().getInfo()
print(f"\n📋 Total bands with GGFE features: {len(band_names)}")
print(f"   New GGFE bands: REE_Index, Bastnasite_Index, Enhanced_Carbonate, Sulfur_Index, Mafic_Index")

🔬 Loading GGFE Advanced Spectral Features...
✅ GGFE module loaded successfully!

📊 Calculating Advanced Spectral Indices...
  ✅ REE Index (Rare Earth Elements)
  ✅ Bastnäsite Index (REE carbonates)
  ✅ Enhanced Carbonate Index
  ✅ Sulfur Index (alunite, jarosite)
  ✅ Mafic Index (kimberlite indicator)

📋 Total bands with GGFE features: 31
   New GGFE bands: REE_Index, Bastnasite_Index, Enhanced_Carbonate, Sulfur_Index, Mafic_Index

📋 Total bands with GGFE features: 31
   New GGFE bands: REE_Index, Bastnasite_Index, Enhanced_Carbonate, Sulfur_Index, Mafic_Index


In [21]:
# 11.2 GGFE - Geobotanical Features
# ===================================
print("🌿 Calculating Geobotanical Stress Features...")

geobotanical = GeobotanicalFeatures()

# VIGS - Vegetation Indicative of Geological Stress
composite_ggfe = geobotanical.vigs_index(composite_ggfe)
print("  ✅ VIGS Index (vegetation geological stress)")

# Metal Stress Index
composite_ggfe = geobotanical.metal_stress_index(composite_ggfe)
print("  ✅ Metal Stress Index (Cu, Zn, Pb toxicity)")

# Chlorophyll Anomaly
composite_ggfe = geobotanical.chlorophyll_anomaly(composite_ggfe)
print("  ✅ Chlorophyll Anomaly (red-edge ratio)")

# Soil Brightness 
composite_ggfe = geobotanical.soil_brightness(composite_ggfe)
print("  ✅ Soil Brightness Index (bare ground detection)")

print("\n📊 Geobotanical Features Summary:")
print("   These features detect vegetation stress caused by:")
print("   - Heavy metal toxicity in soil")
print("   - Anomalous soil geochemistry over mineralized zones")
print("   - Reduced chlorophyll in stressed plants")
print("   - Exposed soil/rock surfaces")

🌿 Calculating Geobotanical Stress Features...
  ✅ VIGS Index (vegetation geological stress)
  ✅ Metal Stress Index (Cu, Zn, Pb toxicity)
  ✅ Chlorophyll Anomaly (red-edge ratio)
  ✅ Soil Brightness Index (bare ground detection)

📊 Geobotanical Features Summary:
   These features detect vegetation stress caused by:
   - Heavy metal toxicity in soil
   - Anomalous soil geochemistry over mineralized zones
   - Reduced chlorophyll in stressed plants
   - Exposed soil/rock surfaces


In [22]:
# 11.3 GGFE - SAR Texture Features
# ==================================
print("📡 Calculating SAR Texture Features...")

sar_texture = SARTextureFeatures()

# Calculate GLCM texture from SAR VV band
print("\n📊 GLCM Texture Analysis (VV band)...")
try:
    glcm_vv = sar_texture.calculate_glcm(sar_composite, band='VV', size=7)
    print("  ✅ GLCM Contrast - local intensity variations")
    print("  ✅ GLCM Entropy - texture randomness/disorder")
    print("  ✅ GLCM Correlation - linear dependency")
    print("  ✅ GLCM ASM - angular second moment (uniformity)")
except Exception as e:
    print(f"  ⚠️ GLCM calculation skipped: {e}")
    glcm_vv = None

# Depolarization Ratio
depol = sar_texture.depolarization_ratio(sar_composite)
print("  ✅ VH/VV Depolarization Ratio")

# Radar Vegetation Index
rvi = sar_texture.radar_vegetation_index(sar_composite)
print("  ✅ Radar Vegetation Index (RVI)")

# Surface Roughness
roughness = sar_texture.surface_roughness_index(sar_composite)
print("  ✅ Surface Roughness Index")

# SAR Lineament Enhancement
sar_lineaments_enhanced = sar_texture.sar_lineament_enhancement(sar_composite)
print("  ✅ SAR Lineament Enhancement")

print("\n📡 SAR Features Interpretation:")
print("   - High GLCM Contrast → rough/fractured surfaces")
print("   - High Entropy → complex geology")
print("   - Low RVI → bare rock/exposed soil")
print("   - High Roughness → structural complexity")

📡 Calculating SAR Texture Features...

📊 GLCM Texture Analysis (VV band)...
  ✅ GLCM Contrast - local intensity variations
  ✅ GLCM Entropy - texture randomness/disorder
  ✅ GLCM Correlation - linear dependency
  ✅ GLCM ASM - angular second moment (uniformity)
  ✅ VH/VV Depolarization Ratio
  ✅ Radar Vegetation Index (RVI)
  ✅ Surface Roughness Index
  ✅ SAR Lineament Enhancement

📡 SAR Features Interpretation:
   - High GLCM Contrast → rough/fractured surfaces
   - High Entropy → complex geology
   - Low RVI → bare rock/exposed soil
   - High Roughness → structural complexity


In [23]:
# 11.4 GGFE - Crosta PCA Method
# ===============================
print("🎯 Applying Crosta Directed PCA Method...")

crosta = CrostaPCA()

# Iron PCA - enhances iron oxide minerals
print("\n📊 Iron PCA (B2, B3, B4, B8)...")
try:
    iron_pca = crosta.iron_pca(composite, roi)
    print("  ✅ Fe_PC1, Fe_PC2, Fe_PC3, Fe_PC4 computed")
    print("  📍 High Fe_PC loading = hematite, goethite enrichment")
except Exception as e:
    print(f"  ⚠️ Iron PCA skipped: {e}")
    iron_pca = None

# Hydroxyl PCA - enhances clay minerals
print("\n📊 Hydroxyl PCA (B8, B8A, B11, B12)...")
try:
    hydroxyl_pca = crosta.hydroxyl_pca(composite, roi)
    print("  ✅ OH_PC1, OH_PC2, OH_PC3, OH_PC4 computed")
    print("  📍 High OH_PC loading = kaolinite, illite, montmorillonite")
except Exception as e:
    print(f"  ⚠️ Hydroxyl PCA skipped: {e}")
    hydroxyl_pca = None

print("\n🔬 Crosta PCA Interpretation:")
print("   - PC with high B4 + negative B3 loading → Iron oxides")
print("   - PC with high B11/B12 contrast → Clay minerals")
print("   - Eigenvector signs indicate mineral presence/absence")

🎯 Applying Crosta Directed PCA Method...

📊 Iron PCA (B2, B3, B4, B8)...
  ✅ Fe_PC1, Fe_PC2, Fe_PC3, Fe_PC4 computed
  📍 High Fe_PC loading = hematite, goethite enrichment

📊 Hydroxyl PCA (B8, B8A, B11, B12)...
  ✅ OH_PC1, OH_PC2, OH_PC3, OH_PC4 computed
  📍 High OH_PC loading = kaolinite, illite, montmorillonite

🔬 Crosta PCA Interpretation:
   - PC with high B4 + negative B3 loading → Iron oxides
   - PC with high B11/B12 contrast → Clay minerals
   - Eigenvector signs indicate mineral presence/absence


In [24]:
# 11.5 GGFE - Mineral Prospectivity Pipelines
# ============================================
print("🎯 Running Mineral-Specific Prospectivity Pipelines...")

pipelines = MineralPipelines()

# Ensure NDVI exists on composite
if 'NDVI' not in composite.bandNames().getInfo():
    ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
    composite_with_ndvi = composite.addBands(ndvi)
else:
    composite_with_ndvi = composite

# Get DEM for pipelines
dem = ee.Image('USGS/SRTMGL1_003').clip(roi)

# 1. Gold Orogenic Pipeline
print("\n⛏️ Gold Orogenic Pipeline (Shear-hosted deposits)...")
gold_orogenic = pipelines.gold_orogenic_pipeline(composite_with_ndvi, dem)
print("  ✅ Gold_Orogenic_Score computed")
print("  📍 Targets: quartz veins, phyllic alteration, structural complexity")

# 2. Gold Epithermal Pipeline
print("\n⛏️ Gold Epithermal Pipeline (Hot spring deposits)...")
gold_epithermal = pipelines.gold_epithermal_pipeline(composite_with_ndvi)
print("  ✅ Gold_Epithermal_Score computed")
print("  📍 Targets: acid-sulfate alteration, silicification")

# 3. Kimberlite/Diamond Pipeline
print("\n💎 Kimberlite Pipeline (Diamond pipes)...")
kimberlite = pipelines.kimberlite_pipeline(composite_with_ndvi, dem)
print("  ✅ Kimberlite_Score computed")
print("  📍 Targets: ultramafic composition, topographic depressions")

# 4. Ruby/Gemstone Pipeline
print("\n💎 Ruby/Gemstone Pipeline (Marble-hosted, Luc Yen type)...")
ruby = pipelines.ruby_gemstone_pipeline(composite_with_ndvi)
print("  ✅ Ruby_Gemstone_Score computed")
print("  📍 Targets: carbonate rocks, low iron, vegetation stress")

# 5. REE Ion-Adsorption Pipeline
print("\n🔋 REE Ion-Adsorption Pipeline (South China type)...")
ree = pipelines.ree_ionadsorption_pipeline(composite_with_ndvi, dem)
print("  ✅ REE_Score computed")
print("  📍 Targets: weathered granite, clay-rich crusts")

print("\n✅ All mineral prospectivity pipelines executed successfully!")

🎯 Running Mineral-Specific Prospectivity Pipelines...

⛏️ Gold Orogenic Pipeline (Shear-hosted deposits)...
  ✅ Gold_Orogenic_Score computed
  📍 Targets: quartz veins, phyllic alteration, structural complexity

⛏️ Gold Epithermal Pipeline (Hot spring deposits)...
  ✅ Gold_Epithermal_Score computed
  📍 Targets: acid-sulfate alteration, silicification

💎 Kimberlite Pipeline (Diamond pipes)...
  ✅ Kimberlite_Score computed
  📍 Targets: ultramafic composition, topographic depressions

💎 Ruby/Gemstone Pipeline (Marble-hosted, Luc Yen type)...
  ✅ Ruby_Gemstone_Score computed
  📍 Targets: carbonate rocks, low iron, vegetation stress

🔋 REE Ion-Adsorption Pipeline (South China type)...
  ✅ REE_Score computed
  📍 Targets: weathered granite, clay-rich crusts

✅ All mineral prospectivity pipelines executed successfully!

⛏️ Gold Orogenic Pipeline (Shear-hosted deposits)...
  ✅ Gold_Orogenic_Score computed
  📍 Targets: quartz veins, phyllic alteration, structural complexity

⛏️ Gold Epithermal Pi

In [25]:
# 11.6 GGFE - Visualization Map
# ===============================
print("🗺️ Creating GGFE Feature Visualization Map...")

Map3 = geemap.Map(center=[16.0, 107.5], zoom=8)

# Base layers
Map3.add_basemap('SATELLITE')
Map3.addLayer(composite_with_ndvi.select(['B4', 'B3', 'B2']), 
              {'min': 0, 'max': 3000}, 'RGB', False)

# GGFE Feature layers
# Gold Orogenic Score
gold_score = gold_orogenic.select('Gold_Orogenic_Score')
Map3.addLayer(
    gold_score,
    {'min': 0, 'max': 3, 'palette': ['white', 'yellow', 'orange', 'red']},
    '⛏️ Gold Orogenic Score',
    True, 0.7
)

# REE Score  
ree_score = ree.select('REE_Score')
Map3.addLayer(
    ree_score,
    {'min': 0, 'max': 3, 'palette': ['white', 'lightblue', 'blue', 'darkblue']},
    '🔋 REE Score',
    False, 0.7
)

# Ruby/Gemstone Score
ruby_score = ruby.select('Ruby_Gemstone_Score')
Map3.addLayer(
    ruby_score,
    {'min': 0, 'max': 3, 'palette': ['white', 'pink', 'magenta', 'darkviolet']},
    '💎 Ruby Score',
    False, 0.7
)

# VIGS (vegetation stress)
vigs = composite_ggfe.select('VIGS')
Map3.addLayer(
    vigs,
    {'min': -0.5, 'max': 0.5, 'palette': ['blue', 'white', 'red']},
    '🌿 VIGS Stress',
    False, 0.7
)

# SAR Roughness
Map3.addLayer(
    roughness,
    {'min': -5, 'max': 15, 'palette': ['black', 'gray', 'white']},
    '📡 SAR Roughness',
    False, 0.7
)

# ROI boundary
Map3.addLayer(roi, {'color': 'cyan'}, 'ROI')

print("✅ GGFE Visualization Map Ready!")
print("\n📊 Available GGFE Layers:")
print("  1. ⛏️ Gold Orogenic Score (shear-hosted)")
print("  2. 🔋 REE Score (ion-adsorption)")
print("  3. 💎 Ruby Score (marble-hosted)")
print("  4. 🌿 VIGS Vegetation Stress")
print("  5. 📡 SAR Surface Roughness")

Map3

🗺️ Creating GGFE Feature Visualization Map...
✅ GGFE Visualization Map Ready!

📊 Available GGFE Layers:
  1. ⛏️ Gold Orogenic Score (shear-hosted)
  2. 🔋 REE Score (ion-adsorption)
  3. 💎 Ruby Score (marble-hosted)
  4. 🌿 VIGS Vegetation Stress
  5. 📡 SAR Surface Roughness
✅ GGFE Visualization Map Ready!

📊 Available GGFE Layers:
  1. ⛏️ Gold Orogenic Score (shear-hosted)
  2. 🔋 REE Score (ion-adsorption)
  3. 💎 Ruby Score (marble-hosted)
  4. 🌿 VIGS Vegetation Stress
  5. 📡 SAR Surface Roughness


Map(center=[16.0, 107.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

## 12. GGFE Framework Summary

### 🔬 Features Implemented

| Category | Feature | Formula/Method | Application |
|----------|---------|----------------|-------------|
| **Advanced Spectral** | REE Index | B5/B6 | Rare Earth detection |
| | Bastnäsite Index | (B11-B12)/(B11+B12) × (B4/B3) | REE carbonates |
| | Enhanced Carbonate | (B11×B12)/(B8²) | Marble/limestone |
| | Sulfur Index | (B4+B12)/(B3+B11) | Acid-sulfate alteration |
| | Mafic Index | (B11×B8)/(B12×B4) | Kimberlite indicator |
| **Geobotanical** | VIGS | [(G-R)/(G+R)] × (NIR/SWIR) | Geological stress |
| | Metal Stress | (R/G) × (SWIR/NIR) | Heavy metal toxicity |
| | Chlorophyll Anomaly | (B5-B4)/(B5+B4) | Vegetation health |
| **SAR Texture** | GLCM Metrics | 7×7 window | Surface roughness |
| | Depolarization | VH/VV linear | Scattering mechanism |
| | RVI | 4×VH/(VV+VH) | Vegetation density |
| **Crosta PCA** | Iron PCA | B2,B3,B4,B8 | Fe oxide enhancement |
| | Hydroxyl PCA | B8,B8A,B11,B12 | Clay mineral enhancement |

### 🎯 Mineral Pipelines

| Pipeline | Primary Target | Key Features |
|----------|----------------|--------------|
| **Gold Orogenic** | Shear-hosted gold | PAI, Silica, TPI |
| **Gold Epithermal** | Hot spring Au | AAA, Sulfur, Thermal |
| **Kimberlite** | Diamond pipes | Mafic, Ferrous, Depression |
| **Ruby/Gemstone** | Marble-hosted | Carbonate, Low Fe, VIGS |
| **REE Ion-Adsorption** | South China type | REE, Bastnäsite, Clay |

### ⚠️ Important Notes

- **Vegetation mask** applied at NDVI thresholds (0.5-0.7)
- **Laterite discrimination** to avoid false positives
- **Structural integration** using DEM-derived features
- **Multi-sensor fusion** for robust detection